In [ ]:
import datetime
import os

import pandas as pd

import telegram
import asyncio
import matplotlib.pyplot as plt
import sql.get_table
import config.sql_queries
from datetime import datetime, timedelta
import tools.clean_processes
from tools.utils import sync_timed

import sys
import logging

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.DEBUG)
handler.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))

logger.addHandler(handler)

engine = sql.get_table.engine
loaded_candles = None


@sync_timed()
def load_candles():
    global loaded_candles
    if loaded_candles is None:
        logger.info("load candles: getting df_all_candles_t candles from database")
        loaded_candles = sql.get_table.query_to_df(
            f"select * from df_all_candles_t  where datetime >  (CURRENT_DATE-14) order by datetime asc")
    else:
        logger.info("load candles: getting df_all_candles_t candles from cache")
    return loaded_candles



In [ ]:
def get_gains(min_lag=10, threshold=0.5, base_asset='MXZ3'):
    # возвращаем то чот выросло нв трешхолд процентов за минлаг минут
    df = load_candles()
    df['cdate'] = pd.to_datetime(df['datetime'])  # , format="%d.%m.%Y %H:%M")

    start_date = datetime.now() - timedelta(minutes=min_lag)
    start_date = start_date.replace(tzinfo=df['cdate'][0].tzinfo)

    logger.info(f"get_gains:df {start_date} + {df['cdate'][0].tzinfo} + \n{df.head()}")
    df_start = df[df['cdate'] < start_date]
    df_end = df

    df_start = df_start.sort_values(['security', 'cdate']).groupby(['security']).tail(1)
    df_end = df_end.sort_values(['security', 'cdate']).groupby(['security']).tail(1)

    df_res = df_end.merge(df_start, how='inner', on='security')[
        ['security', 'class_code_x', 'close_x', 'close_y', 'cdate_x', 'cdate_y']]
    df_res['inc'] = (df_res['close_x'] / df_res['close_y'] - 1) * 100
    
    df_betas = sql.get_table.query_to_df(f"SELECT sec, beta, r2, corr FROM public.analytics_beta where base_asset = '{base_asset}'")
    base_inc = df_res[df_res['security']==base_asset]['inc'].iloc[0] 
    df_res['base_inc'] = base_inc
    df_res = df_res.merge(df_betas, left_on='security', right_on='sec', how='left')
    df_res.drop('sec', axis=1, inplace=True)  
    logger.info(f"get_gains:df_res \n {df_res.head()}")
    return df_res

In [ ]:
df_res = get_gains()
df_res

In [ ]:
def get_abnormal_volumes(include_daily=True, minutes_lookback=10, days_lookback=14):
    def get_volumes(minutes_lookback=minutes_lookback, days_lookback=days_lookback):
        start_time = (datetime.now() - timedelta(minutes=minutes_lookback)).time()
        end_time = (datetime.now()).time()

        start_date = (datetime.now() - timedelta(days=days_lookback)).date()
        end_date = (datetime.now()).date()

        df = load_candles()
        df['cdate'] = pd.to_datetime(df['datetime'])
        df['ctime'] = df['cdate'].dt.time
        df['cdt'] = df['cdate'].dt.date

        # считаем mean std volumes предыдущего и текущего периодов
        df_prev = df.loc[(df['cdate'].dt.time > start_time) & (df['cdate'].dt.time <= end_time) &
                         (df['cdate'].dt.date >= start_date) & (df['cdate'].dt.date < end_date)] \
            .groupby([df['cdate'].dt.date, 'security']).sum('volume').reset_index() \
            .groupby('security').agg(volume_mean=('volume', 'mean'), volume_std=('volume', 'std')).reset_index()

        df_now = df.loc[(df['cdate'].dt.time > start_time) & (df['cdate'].dt.time <= end_time) &
                        (df['cdate'].dt.date == end_date)] \
            .groupby([df['cdate'].dt.date, 'security']).sum('volume').reset_index()[['security', 'volume']]

        df_analys = df_prev.merge(df_now, how='inner', on='security')
        df_analys['std'] = (df_analys['volume'] - df_analys['volume_mean']) / df_analys['volume_std']
        df_analys['end_time'] = end_time
        
        df_inc = get_gains(min_lag=minutes_lookback)
        df_analys = df_analys.merge(df_inc[['security', 'inc', 'base_inc', 'beta', r2]], how='left', on='security')

        return df_analys[df_analys['std'] >= 2].sort_values('std', ascending=False)

    df_minutes = get_volumes()
    df_minutes['timeframe'] = 'mins'

    # 540 это -9 часов, чтобы это сработало в 9 утра
    df_daily = get_volumes(minutes_lookback=540) if include_daily else pd.DataFrame()
    df_daily['timeframe'] = 'days'

    return pd.concat([df_minutes, df_daily], axis=0).reset_index()

In [ ]:
get_abnormal_volumes()


In [ ]:
df_betas

In [ ]:
import datetime
import json
import os
import asyncio
import string

from dotenv import load_dotenv
from pyrogram import Client
from pymongo import MongoClient


load_dotenv(dotenv_path='./my.env')

key = os.environ['tg_key']
api_id = os.environ['tg_api_id']
api_hash = os.environ['tg_api_hash']

client = MongoClient()




async def create_record(chat_id):
    async with Client("my_ccount", api_id, api_hash) as app:
        chat = await app.get_chat(chat_id)
        # count = await app.get_chat_history_count(chat_id=chat_id)
        # print(chat)
        chat = json.loads(str(chat))
        result = dict()
        result["tg_id"] = chat['id']
        result['is_active'] = 1
        result["title"] = chat['title'].strip()
        result["username"] = chat.get('username', '').strip()
        result['description'] = chat.get('description', '').strip()
        result['members_count'] = chat['members_count']
        result['count'] = 0  # count - 100 # to import 100 messages after creation
        # print(result)
        return result



names_collection = client.trading['tg_channels']
res = await create_record(-1001642909090)
print(res)
names_collection.insert_one(res)



In [ ]:
    channel={ 'tg_id': -1001642909090, 'is_active': 1, 'title': 'ВИП канал', 'username': 'chekhov_vip', 'description': '', 'members_count': 1901, 'count': 0, 'out_id': 81}
    async with Client("my_ccount_tgchannels", api_id, api_hash) as app:
        news_collection = client.trading['news']

        print(f"\nimporting channel {channel['title']}:\n{channel}")

        if channel is None:
            print("Error: channel id is None")

        chat_id = -1001642909090
        count = await app.get_chat_history_count(chat_id=chat_id)

        new_msg_count = count - channel['count']
        print(f"{channel['username']} has {new_msg_count} new messages")
        if limit is None:
            limit = min(count - channel['count'], max_msg_load)

In [ ]:
df_res

In [ ]:
import datetime

str(int((datetime.datetime.utcnow() - datetime.datetime(2023, 1, 1)).total_seconds() * 1000000) % 1000000000)

In [ ]:
str(int((datetime.datetime.utcnow() - datetime.datetime(2023, 1, 1)).total_seconds() * 1000000) % 1000000000)

In [ ]:
df = df.sort_values('datetime')
df

In [ ]:
df_eq

In [ ]:
df_volumes

In [ ]:
    fig, ax_left = plt.subplots()

    plt.xticks(rotation=90)
    fig.set_figheight(9)
    fig.set_figwidth(16)
    fig.align_ylabels()


    ax_right = ax_left.twiny()
    if len(df_volumes) > 0:
        ax_right.plot(df_volumes['volume'], df_volumes['price'], color='green', linestyle='dashed')
        ax_right.axis(xmax=max(df_volumes['volume']) * 3)

    #ax_left.set_xticklabels(df['datetime'].astype(str))
    ax_left.locator_params(axis='x', nbins=25)
    ax_left.locator_params(axis='y', nbins=20)
    
    #plt.xticks(new_labels)
    ax_left.plot(df['close'])
    # ax_left.plot(df['t'],df['close'])
    res = []
    prev_row = None
    for idx, row in df.iterrows():
        if row['t'][:10] != prev_row:
            res.append((idx, row['t'][:10]))
        prev_row = row['t'][:10]

    for idx,dt in res:
        ax_left.axvline(x=idx, color='g', linestyle='-', label = dt)    
    
    
    plt.title('title')
    for _, row in df_eq.iterrows():  # np.array([t[0] for t in peaks]):
        ax_left.axhline(y=row['price'], color='r', linestyle='-')
        if row['min_start']:
            # ax_left.axhline(y=row['min_start'], color='g', linestyle='-')
            # ax_left.axhline(y=row['max_start'], color='g', linestyle='-')
            # ax_left.axhline(y=row['end'], color='m', linestyle='-')
            # if (row['down']) != "0":
            #    ax_left.axhline(y=row['sl'], color='k', linestyle='-')
            pass

    plt.show()

In [ ]:
res = []
prev_row = None
for idx, row in df.iterrows():
    if row['t'][:10] != prev_row:
        res.append((idx, row['t'][:10]))
    prev_row = row['t'][:10]
    
print(res)

In [ ]:
import sql.get_table
import pandas as pd

engine = sql.get_table.engine

def copy_colvals(df_monitor, colpairs, is_upd_only=False):
    for pairs in colpairs:
        if is_upd_only == False:
            df_monitor.loc[df_monitor[pairs[1]].notnull(), pairs[0]] = df_monitor.loc[
                df_monitor[pairs[1]].notnull(), pairs[1]]
        else:
            df_monitor.loc[df_monitor['to_update'] & df_monitor[pairs[1]].notnull(), pairs[0]] = df_monitor.loc[
                df_monitor['to_update'] & df_monitor[pairs[1]].notnull(), pairs[1]]

    return df_monitor


In [ ]:

    
    df_monitor = []

    try:
        df_monitor = pd.DataFrame(engine.execute(
            "select code, old_state, old_price, old_start, old_end, new_state, new_price, new_start, new_end, std, "
            "old_timestamp, new_timestamp from public.df_monitor"))
    except:
        pass

    check_consistancy_query = """select  code, count(*)	FROM public.df_monitor
        group by code having count(*)>1"""

    if len(df_monitor) == 0 or len(pd.DataFrame(engine.execute(check_consistancy_query))) > 0:
        print("consistancy check failed")
        exit(0)
        
    columns = df_monitor.columns

    query = """select l.code, name as state, price, start, "end", std as new_std, now() as timestamp from df_all_levels l 
    inner join
    (
    select code, (bid + ask)/2 as price from public.futquotes where bid > 0
    union all
    select code, (bid + ask)/2 as price from public.secquotes where bid > 0) as q
    on l.code = q.code where 
    l.start <= q.price and l.end > q.price
    order by l.code desc"""

    df_new = pd.DataFrame(engine.execute(query))
    print("df_new (new data):\n", df_new.head())

    df_monitor = df_monitor.merge(df_new, how='outer', on='code')
    print("df_monitor: moving new state to old state\n", df_monitor.head())

    # переносим not null новое в старое и переносим цену и стд
    colpairs = [('old_price', 'new_price'), ('old_state', 'new_state'), ('old_start', 'new_start'), \
                ('old_end', 'new_end'), ('old_timestamp', 'new_timestamp'), ('new_price', 'price'), ('std', 'new_std'),
                ('new_timestamp', 'timestamp')]

    df_monitor = copy_colvals(df_monitor, colpairs)
    
    
    print("step2: df_monitor\n", df_monitor.head())



In [ ]:
    df_monitor

In [ ]:
    df_monitor['to_update'] = df_monitor['new_state'].isnull() | (
            df_monitor['new_price'] + df_monitor['std'] < df_monitor['old_start']) | \
                              (df_monitor['new_price'] - df_monitor['std'] > df_monitor['old_end'])

    df_monitor[df_monitor['to_update']]

 pid_process(caller):[(73459, 'python', 0.14731385000000002)]
False: 0.14746954999999998 0.1639805166666667
False: 0.14747470000000001 0.1639805166666667
False: 0.14731385000000002 0.1639805166666667
state: True
urgent_list: ['SRU3', 'CRU3', 'SUR', 'CHMF', 'LKOH', 'MAGN', 'MGNT', 'NLMK', 'PLZL', 'RUAL', 'VTBR']
df_new (new data):
     code     state  ...     new_std                        timestamp
0   YNU3  new_high  ...   31.603675 2023-06-15 10:03:10.089221+03:00
1   VTBR  tp_start  ...    0.000074 2023-06-15 10:03:10.089221+03:00
2   VBU3     sl_tp  ...    2.418547 2023-06-15 10:03:10.089221+03:00
3   UPRO   observe  ...    0.004206 2023-06-15 10:03:10.089221+03:00
4  TRNFP  new_high  ...  368.668529 2023-06-15 10:03:10.089221+03:00

[5 rows x 7 columns]
df_monitor: moving new state to old state
    code old_state  ...    new_std                        timestamp
0  YNU3  new_high  ...  31.603675 2023-06-15 10:03:10.089221+03:00
1  VBU3     sl_tp  ...   2.418547 2023-06-15 10:03:10.089221+03:00
2  SRU3   observe  ...  12.259940 2023-06-15 10:03:10.089221+03:00
3  SiU3   observe  ...  23.603897 2023-06-15 10:03:10.089221+03:00
4  RNU3   observe  ...  61.082371 2023-06-15 10:03:10.089221+03:00

[5 rows x 18 columns]
step2: df_monitor
    code old_state  ...    new_std                        timestamp
0  YNU3  new_high  ...  31.603675 2023-06-15 10:03:10.089221+03:00
1  VBU3     sl_tp  ...   2.418547 2023-06-15 10:03:10.089221+03:00
2  SRU3   observe  ...  12.259940 2023-06-15 10:03:10.089221+03:00
3  SiU3   observe  ...  23.603897 2023-06-15 10:03:10.089221+03:00
4  RNU3   observe  ...  61.082371 2023-06-15 10:03:10.089221+03:00

[5 rows x 18 columns]
step3: df_monitor full (updated states)
    code old_state  ...                        timestamp  to_update
0  YNU3  new_high  ... 2023-06-15 10:03:10.089221+03:00      False
1  VBU3     sl_tp  ... 2023-06-15 10:03:10.089221+03:00      False
2  SRU3   observe  ... 2023-06-15 10:03:10.089221+03:00      False
3  SiU3   observe  ... 2023-06-15 10:03:10.089221+03:00       True
4  RNU3   observe  ... 2023-06-15 10:03:10.089221+03:00      False

[5 rows x 19 columns]
step3: df_monitor[to_update]==True - filtered
      code old_state  ...                        timestamp  to_update
3    SiU3   observe  ... 2023-06-15 10:03:10.089221+03:00       True
8    NKU3     sl_tp  ... 2023-06-15 10:03:10.089221+03:00       True
15   EuU3   observe  ... 2023-06-15 10:03:10.089221+03:00       True
46   NLMK   observe  ... 2023-06-15 10:03:10.089221+03:00       True
71  BANEP     start  ... 2023-06-15 10:03:10.089221+03:00       True
72   AQUA   observe  ... 2023-06-15 10:03:10.089221+03:00       True

[6 rows x 19 columns]
df_monitor finished
df_monitor      code old_state  ...                        timestamp  to_update
3    SiU3   observe  ... 2023-06-15 10:03:10.089221+03:00       True
8    NKU3     sl_tp  ... 2023-06-15 10:03:10.089221+03:00       True
15   EuU3   observe  ... 2023-06-15 10:03:10.089221+03:00       True
46   NLMK   observe  ... 2023-06-15 10:03:10.089221+03:00       True
71  BANEP     start  ... 2023-06-15 10:03:10.089221+03:00       True

[5 rows x 19 columns]
3      SiU3
8      NKU3
15     EuU3
46     NLMK
71    BANEP
72     AQUA
Name: code, dtype: object


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# https://www.geeksforgeeks.org/how-to-schedule-python-scripts-as-cron-jobs-with-crontab/
import asyncio
import os

import pandas as pd
import numpy as np
import config.sql_queries
from datetime import datetime, timedelta
from scipy.signal import find_peaks

import time
import sql.get_table
import telegram
import pytz

import tools.clean_processes

engine = sql.get_table.engine


def load_df(days_to_subtract=7):
    df = sql.get_table.query_to_df("select * from df_all_candles_t where datetime > now() - interval '14 days'")
    df['t'] = pd.to_datetime(df['datetime'])
    df.drop(columns=['datetime'], inplace=True)

    df['week'] = df['t'].dt.isocalendar().week
    df['day'] = df['t'].dt.day
    current_week = datetime.today().isocalendar().week
    current_day = datetime.today()

    df = df[(df['volume'] > 0) & (df['close'] > 0)]

    # Adjust start date
    start_date = (datetime.today() - timedelta(days=days_to_subtract))
    start_date = start_date.replace(tzinfo=df['t'][0].tzinfo)
    df = df[df['t'] > start_date]

    # adjust past week volume
    df.loc[df['week'] != current_week, 'volume'] = df.loc[df['week'] != current_week, 'volume'] / 2
    df.loc[df['day'] == current_day, 'volume'] = df.loc[df['day'] == current_day, 'volume'] * 2
    return df


def build_levels(df_):
    df_levels = pd.DataFrame()
    df_all_levels = pd.DataFrame()
    df_all_volumes = pd.DataFrame()

    for eq in df_['security'].drop_duplicates():

        df = df_[df_['security'] == eq]
        df = df.reset_index()
        std = np.std((df['close'] - df['close'].shift(1)))
        print(eq, std)
        np_close = np.array(df['close'])

        price_range = []
        volumes = []
        mult = 100

        for x in np.linspace(np.min(np_close), np.max(np_close), mult):
            price_range.append(x)
            volumes.append(np.dot(df['volume'], ((np_close > x - std) & (np_close < x + std))))
        volumes = np.convolve(volumes, np.ones(10), mode='same')

        idx, _ = find_peaks(volumes)
        peaks = list(zip([price_range[t] for t in idx], [volumes[t] for t in idx]))

        peaks.insert(0, (np.min(np_close), 0))
        peaks.insert(len(peaks), (np.max(np_close), 0))

        for _ in range(1, len(peaks)):
            for i in range(1, len(peaks)):
                if peaks[i][0] - peaks[i - 1][0] < 2 * std:
                    # print(peaks)
                    if peaks[i][1] < peaks[i - 1][1]:
                        # print("del", i)
                        del peaks[i]
                    else:
                        # print("del", i - 1)
                        del peaks[i - 1]
                    # print("after:", peaks)
                    break

        # print(peaks)

        df_eq = pd.DataFrame(peaks, columns=['price', 'volume'])
        df_eq['std'] = std
        df_eq['sec'] = eq
        df_eq[['min_start', 'max_start', 'end', 'sl', 'mid', 'down', 'prev_end', 'next_sl']] = None  ##

        max_level = 0.3
        close_level = 0.9
        sl_level = 0.8

        # build levels
        for idx, row in df_eq.iterrows():
            if idx >= 1:  # >= если хотим ловить падающий нож
                df_eq.loc[idx, 'mid'] = df_eq.loc[idx - 1, 'price'] if idx >= 1 else 0
                df_eq.loc[idx, 'down'] = df_eq.loc[idx - 2, 'price'] if idx >= 2 else 0

                min_start = df_eq.loc[idx, 'mid'] + row['std']
                max_start = df_eq.loc[idx, 'mid'] + (row['price'] - df_eq.loc[idx, 'mid']) * max_level
                sl = min((sl_level - 1) * (df_eq.loc[idx, 'mid'] - df_eq.loc[idx, 'down']) + df_eq.loc[idx, 'mid'],
                         df_eq.loc[idx, 'mid'] - 2 * row['std'])

                if min_start < max_start:
                    df_eq.loc[idx, 'min_start'] = min_start
                    df_eq.loc[idx, 'max_start'] = max_start
                    df_eq.loc[idx, 'end'] = df_eq.loc[idx, 'mid'] + (row['price'] - df_eq.loc[idx, 'mid']) * close_level
                    df_eq.loc[idx, 'sl'] = sl

        # fill prev-next sl
        for idx, row in df_eq.iterrows():
            df_eq.loc[idx, 'prev_end'] = (df_eq.loc[idx - 1, 'end'] if idx - 1 >= 0 else None)
            df_eq.loc[idx, 'next_sl'] = (df_eq.loc[idx + 1, 'sl'] if idx + 1 < len(df_eq) else None)

        # creating price-volume df
        df_price_volume = pd.DataFrame({"price": price_range, "volume": volumes})
        df_price_volume['code'] = eq
        # print(df_price_volume)

        # create 1 level 1 row table
        df_eq_all_levels = create_all_levels(df_eq)
        df_eq_all_levels['std'] = std
        df_eq_all_levels = compress_all_levels(df_eq_all_levels)

        df_levels = pd.concat([df_levels, df_eq])
        df_all_levels = pd.concat([df_all_levels, df_eq_all_levels])
        df_all_volumes = pd.concat([df_all_volumes, df_price_volume])

        df_levels['implied_prob'] = ((df_levels['min_start'] + df_levels['max_start']) / 2 - df_levels['sl']) / (
                df_levels['end'] - df_levels['sl'])
    return df_levels, df_all_levels, df_all_volumes


def create_all_levels(df_levels):
    df_all_levels = pd.DataFrame([], columns=['code', 'name', 'start', 'end', 'logic'])
    for idx, row in df_levels.iterrows():
        if idx == 0:
            # print(row['next_sl'])
            if row['next_sl']:
                df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'new_low', 0, row['next_sl'], 0)
            else:
                df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'new_low', 0, row['price'], 0)

        if (not row['prev_end']) and (not row['sl']) and (not row['next_sl']):
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'observe', row['mid'], row['price'], 1)

        elif (not row['prev_end']) and (not row['sl']) and (row['next_sl']):
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'observe', row['mid'], row['next_sl'], 2)

        elif (not row['prev_end']) and (row['sl']) and (not row['next_sl']):
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'sl_start', row['sl'], row['min_start'], 3)
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'start', row['min_start'], row['max_start'], 3)
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'observe', row['max_start'], row['end'], 3)

        elif (not row['prev_end']) and (row['sl']) and (row['next_sl']):
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'sl_start', row['sl'], row['min_start'], 4)
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'start', row['min_start'], row['max_start'], 4)
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'observe', row['max_start'], row['next_sl'], 4)
            # df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'sl_tp', row['next_sl'],row['end'],4)

        elif (row['prev_end']) and (not row['sl']) and (not row['next_sl']):
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'observe', row['prev_end'], row['price'], 5)

        elif (row['prev_end']) and (not row['sl']) and (row['next_sl']):
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'observe', row['prev_end'], row['next_sl'], 6)

        elif (row['prev_end']) and (row['sl']) and (not row['next_sl']):
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'sl_tp', row['sl'], row['prev_end'], 7)
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'tp_start', row['prev_end'], row['min_start'], 7)
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'start', row['min_start'], row['max_start'], 7)
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'observe', row['max_start'], row['end'], 7)

        elif (row['prev_end']) and (row['sl']) and (row['next_sl']):
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'sl_tp', row['sl'], row['prev_end'], 8)
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'tp_start', row['prev_end'], row['min_start'], 8)
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'start', row['min_start'], row['max_start'], 8)
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'observe', row['max_start'], row['next_sl'], 8)
            # df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'sl_tp', row['next_sl'], row['end'],8)

        else:
            pass

        if idx == len(df_levels) - 1:
            if row['sl']:
                df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'tp_new_high', row['end'], row['price'], 9)
            df_all_levels.loc[len(df_all_levels)] = (row['sec'], 'new_high', row['price'], 999999, 9)

    return df_all_levels


def compress_all_levels(df_all_levels):
    for idx in reversed(df_all_levels.index):
        if idx >= 1:
            if not df_all_levels.loc[idx, 'start']:
                df_all_levels.drop(idx, inplace=True)
            elif df_all_levels.loc[idx, 'name'] == df_all_levels.loc[idx - 1, 'name'] \
                    and df_all_levels.loc[idx, 'start'] == df_all_levels.loc[idx - 1, 'end']:
                df_all_levels.loc[idx - 1, 'end'] = df_all_levels.loc[idx, 'end']
                df_all_levels.drop(idx, inplace=True)
            elif df_all_levels.loc[idx, 'start'] != df_all_levels.loc[idx - 1, 'end']:
                print(f'INCONSISTENCY!!! {df_all_levels.iloc[[idx]]} {df_all_levels.iloc[[idx - 1]]}')
    df_all_levels['start'] = pd.to_numeric(df_all_levels['start'])
    df_all_levels['end'] = pd.to_numeric(df_all_levels['end'])
    return df_all_levels



In [ ]:
    df = load_df()
    df

In [ ]:
    df_levels, df_all_levels, df_all_volumes = build_levels(df)


In [ ]:
df_levels

In [ ]:
df_all_volumes[df_all_volumes['code'] == 'RIM3']

In [ ]:
df_all_levels['name'].unique()

In [ ]:
df_all_levels[df_all_levels['name']=='EDU3']

In [ ]:
from pymongo import MongoClient
from bson.objectid import ObjectId
import pandas as pd

client = MongoClient()

In [ ]:
    import datetime
    news_collection = client.trading['news']

    res = pd.DataFrame()

    for item in news_collection.find({'tags':'PLZL','date':{'$gt':datetime.datetime(2023,4,10)}}):#{'tags':'NLMK'}):
        add = pd.DataFrame({'id': item['_id'], 'date': item['date'], 'username': item['channel_username'], 'tags':str(item['tags'])}, index=[0])
        res = pd.concat([res, add])


In [ ]:
res['dt'] =  pd.to_datetime(res['date']).dt.date 
res['wd'] = pd.to_datetime(res['date']).dt.weekday
res = res[res['wd'] <5]

In [ ]:
res.groupby('dt')['date'].count().plot()

In [ ]:
pd.set_option('display.max_rows', None)
res.groupby('dt')['date'].count()

In [ ]:
res

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained("cointegrated/LaBSE-en-ru")
model = AutoModel.from_pretrained("cointegrated/LaBSE-en-ru")
sentences = ["""Минобороны России сообщает, что с начала проведения спецоперации более 10 тысяч военнослужащих ВС РФ получили специальные выплаты за личное уничтожение или захват военной техники противника.

В 2022 году за личное уничтожение 11 586 единиц украинской военной техники и вооружения соответствующие выплаты получили 7 064 военных.

С 1 января по 31 мая этого года за лично уничтоженные 4 415 единиц украинской и западной военной техники получили выплаты 3 193 военнослужащих.

Выплаты производятся на основания приказов командиров воинских частей и перечисляются на личные счета военных. При этом никаких ограничений на получение выплат от количества уже уничтоженных ими ранее единиц техники противника не существует."""]
encoded_input = tokenizer(sentences, padding=True, truncation=True, max_length=64, return_tensors='pt')
with torch.no_grad():
    model_output = model(**encoded_input)
embeddings = model_output.pooler_output
embeddings = torch.nn.functional.normalize(embeddings)
print(embeddings.shape, embeddings)